# CountZES — Run Pipeline on FSC-147 (Subsampled / Full with Multi-Stage Evidence)

Notebook này dùng để chạy CountZES trên tập **FSC-147** (`test` 1,190 ảnh / `val` 1,286 ảnh, 29 classes), hỗ trợ:
- Chọn giới hạn số lượng mẫu (`MAX_IMAGES`, ví dụ `30` / `58` / `0` = full) và cách lấy mẫu (`stratified` rải đều qua các class, `random`, hoặc `first`).
- Lưu đầy đủ **log & evidence nhiều bước** lên Google Drive (`MyDrive/CountZES/outputs/<OUTDIR>/`):
  - `run_config.json`, `summary.json` (kèm `per_class_metrics` theo từng class)
  - `predictions.csv` & `predictions.jsonl` (telemetry từng ảnh qua Stage 1/2/3)
  - `logs/resource_usage.log` & `logs/resource_usage.jsonl` (CPU, RAM, GPU VRAM)
  - `stage_samples/*.png` (lưới 2x4 từ CLIP -> GDINO+SSES -> DAE -> P2P -> GGES/KDE -> DGE -> FRES 2-Means -> FCE -> Final)
  - `FSC147_<split>/refined_*.png` (ảnh density map cuối cùng nếu bật `SAVE_FINAL_VIS = True`)

In [ ]:
# [CELL-1] Kiểm tra GPU & thiết lập biến môi trường PyTorch
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# [CELL-2] Clone hoặc cập nhật code mới nhất từ GitHub (hỗ trợ Public & Private PAT)
import getpass, subprocess

REPO_OWNER_PATH = "github.com/duylw/cv-advanced.git"
REPO_URL = f"https://{REPO_OWNER_PATH}"
REPO_DIR = "/content/cv-advanced"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], env=env_no_prompt)
    if res.returncode != 0:
        print("Repo đang ở chế độ Private -> nhập GitHub PAT (không lưu vào file):")
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "clone", f"https://{token}@{REPO_OWNER_PATH}", REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], env=env_no_prompt)
    if res.returncode != 0:
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "-C", REPO_DIR, "pull", f"https://{token}@{REPO_OWNER_PATH}", "main"], check=True)

%cd {REPO_DIR}
!git log -n 2 --oneline

In [ ]:
# [CELL-3] Cài các thư viện cần thiết (bỏ qua nếu đã có) & kiểm tra syntax
!python -c "import addict, yapf, pycocotools, termcolor, yaml, gdown, supervision, timm, einops, transformers, psutil" 2>/dev/null || pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools termcolor pyyaml gdown psutil "transformers>=4.31.0"
!python -c "import segment_anything" 2>/dev/null || pip install -q git+https://github.com/facebookresearch/segment-anything.git

!python -m py_compile resource_monitor.py countzes_core.py countzes_fsc147.py

In [ ]:
# [CELL-4] Mount Google Drive & liên kết data/weights/outputs
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CountZES"
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data")
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "weights")
DRIVE_OUTPUTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs")

for d in [DRIVE_PROJECT_DIR, DRIVE_DATA_DIR, DRIVE_WEIGHTS_DIR, DRIVE_OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Copy metadata FSC-147 (.json/.txt) từ repo sang Drive nếu Drive chưa có (idempotent)
for root, _, files_in_dir in os.walk("/content/cv-advanced/data", followlinks=False):
    rel_root = os.path.relpath(root, "/content/cv-advanced/data")
    target_root = os.path.join(DRIVE_DATA_DIR, rel_root) if rel_root != "." else DRIVE_DATA_DIR
    os.makedirs(target_root, exist_ok=True)
    for fname in files_in_dir:
        src_f = os.path.join(root, fname)
        dst_f = os.path.join(target_root, fname)
        if not os.path.islink(src_f) and not os.path.exists(dst_f):
            shutil.copy2(src_f, dst_f)

# Symlink thư mục ảnh FSC-147 từ Drive vào ./data/FSC-147/images_384_VarV2
FSC_DRIVE_IMG_DIR = os.path.join(DRIVE_DATA_DIR, "FSC-147", "images_384_VarV2")
os.makedirs(FSC_DRIVE_IMG_DIR, exist_ok=True)
local_fsc_img_dir = os.path.join("data", "FSC-147", "images_384_VarV2")
os.makedirs(os.path.dirname(local_fsc_img_dir), exist_ok=True)
if os.path.islink(local_fsc_img_dir):
    os.unlink(local_fsc_img_dir)
elif os.path.exists(local_fsc_img_dir):
    shutil.rmtree(local_fsc_img_dir)
os.symlink(FSC_DRIVE_IMG_DIR, local_fsc_img_dir)

# Symlink ./outputs -> Drive
if os.path.islink("outputs"):
    os.unlink("outputs")
elif os.path.exists("outputs"):
    shutil.rmtree("outputs")
os.symlink(DRIVE_OUTPUTS_DIR, "outputs")

print("Mounted FSC-147 image dir:", FSC_DRIVE_IMG_DIR)
print("Mounted Outputs dir      :", DRIVE_OUTPUTS_DIR)

In [ ]:
# [CELL-5] Kiểm tra / tải Model Weights vào Drive và symlink
os.makedirs("DSALVANet/checkpoints", exist_ok=True)
os.makedirs("GroundingDINO/weights", exist_ok=True)
os.makedirs("CounTR/output_allnew_dir", exist_ok=True)

def ensure_weight(weight_name, local_path, download_cmd=None):
    drive_path = os.path.join(DRIVE_WEIGHTS_DIR, weight_name)
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        if download_cmd:
            print(f"[Download -> Drive] {drive_path}")
            get_ipython().system(download_cmd.format(dst=drive_path))
        else:
            print(f"[Optional] Chưa có {weight_name} trong {DRIVE_WEIGHTS_DIR}")
            return
    else:
        print(f"[Drive OK] {drive_path}")
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        print(f"[WARNING] Chưa tải được {weight_name}. Đặt thủ công vào: {drive_path}")
        return
    if os.path.islink(local_path) or os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)
    print(f"  -> Linked {local_path}")

ensure_weight(
    "sam_vit_h_4b8939.pth",
    "data/sam_vit_h_4b8939.pth",
    "wget -q --show-progress -O '{dst}' https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
)
ensure_weight(
    "checkpoint_200.pth",
    "DSALVANet/checkpoints/checkpoint_200.pth",
    "gdown 1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp -O '{dst}'"
)
ensure_weight(
    "groundingdino_swint_ogc.pth",
    "GroundingDINO/weights/groundingdino_swint_ogc.pth",
    "gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O '{dst}'"
)
ensure_weight("FSC147.pth", "CounTR/output_allnew_dir/FSC147.pth")

In [ ]:
# [CELL-6] Chuẩn bị ảnh FSC-147 (images_384_VarV2) vào Google Drive (idempotent)
import glob, json, zipfile

FSC147_GDRIVE_ID = "1ymDYrGs9DSRicfZbSCDiOu0ikGDh5k6S"  # Link chính thức từ LearningToCountEverything
ZIP_PATH = "/content/FSC147_images_384_VarV2.zip"
TMP_EXTRACT = "/content/_tmp_fsc147"

existing_imgs = [f for f in os.listdir(FSC_DRIVE_IMG_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
if len(existing_imgs) >= 1000:
    print(f"[Drive OK] Đã có sẵn {len(existing_imgs)} ảnh trong {FSC_DRIVE_IMG_DIR}")
else:
    # Nếu bạn đã tải sẵn file zip lên Drive: tự tìm trước khi tải bằng gdown
    drive_zip_cand = os.path.join(DRIVE_DATA_DIR, "FSC-147", "FSC147_384_V2.zip")
    if os.path.exists(drive_zip_cand) and os.path.getsize(drive_zip_cand) > 0:
        ZIP_PATH = drive_zip_cand
    elif not (os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 0):
        print("Đang tải FSC-147 images_384_VarV2 qua gdown...")
        get_ipython().system(f"gdown {FSC147_GDRIVE_ID} -O '{ZIP_PATH}'")

    if not (os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH)):
        raise RuntimeError(
            f"Không tải/đọc được zip FSC-147 tại {ZIP_PATH} (gdown có thể bị giới hạn quota Google Drive). "
            f"Hãy tải từ https://drive.google.com/file/d/{FSC147_GDRIVE_ID}/view rồi giải nén ảnh vào {FSC_DRIVE_IMG_DIR}"
        )

    shutil.rmtree(TMP_EXTRACT, ignore_errors=True)
    print("Đang giải nén vào /content (nhanh hơn ghi trực tiếp qua FUSE)...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(TMP_EXTRACT)

    extracted_imgs = [
        p for p in glob.glob(os.path.join(TMP_EXTRACT, "**", "*"), recursive=True)
        if p.lower().endswith((".jpg", ".jpeg", ".png")) and "__MACOSX" not in p and not os.path.basename(p).startswith("._")
    ]
    assert len(extracted_imgs) > 0, f"Không tìm thấy ảnh trong {ZIP_PATH}"
    print(f"Tìm thấy {len(extracted_imgs)} ảnh, đang copy sang Drive ({FSC_DRIVE_IMG_DIR})...")
    os.makedirs(FSC_DRIVE_IMG_DIR, exist_ok=True)
    for src_p in extracted_imgs:
        dst_p = os.path.join(FSC_DRIVE_IMG_DIR, os.path.basename(src_p))
        if not os.path.exists(dst_p):
            shutil.copy2(src_p, dst_p)
    shutil.rmtree(TMP_EXTRACT, ignore_errors=True)

# Kiểm tra độ phủ so với test.json
test_meta = [json.loads(l) for l in open(os.path.join(DRIVE_DATA_DIR, "FSC-147", "test.json"), encoding="utf-8") if l.strip()]
present = sum(1 for e in test_meta if os.path.exists(os.path.join(FSC_DRIVE_IMG_DIR, e["filename"])))
print(f"[Dataset Check] Khớp {present}/{len(test_meta)} ảnh trong test.json")
assert present == len(test_meta), f"Thiếu ảnh FSC-147 test split ({present}/{len(test_meta)})"

In [ ]:
# [CELL-7] Chạy pipeline CountZES trên FSC-147 với tùy chọn số lượng mẫu & logging nhiều bước
SPLIT = "test"                 # "test" (1,190 ảnh) hoặc "val" (1,286 ảnh)
FSOC = "DSALVANet"             # Đã có sẵn checkpoint_200.pth

# --- Tùy chọn lấy mẫu ---
MAX_IMAGES = 2               # Đặt 0 để chạy FULL 1,190 ảnh; đặt > 0 (vd: 30, 58, 100) để chạy tập mẫu
SAMPLE_MODE = "stratified"     # "stratified" (chia đều theo 29 class) | "random" | "first"
SAMPLE_SEED = 42               # Seed cố định để tái lập đúng danh sách mẫu khi so sánh nhiều cấu hình

# --- Tùy chọn lưu bằng chứng & log ---
STAGE_SAMPLES = 15             # Số ảnh lưu lưới 2x4 (S1a->S3b->Final); đặt -1 để lưu TẤT CẢ mẫu đã chạy
SAVE_FINAL_VIS = True          # True -> lưu thêm ảnh density map cuối vào outputs/<OUTDIR>/FSC147_<split>/

# --- Preset cấu hình CountZES ---
SSES_MODE = "paper"            # "paper" (khớp paper: box=0.15, p=90, không đảo) | "github" (box=0.20, p=80, có 1-sim)
CLIP_SIM_MODE = "maskclip"     # "raw" (last_hidden_state gốc) | "maskclip" (Value projection bỏ qua Q-K layer cuối)
CLIP_NO_CROP = True            # True (resize đủ toàn ảnh) | False (center-crop vuông mặc định của CLIP)

n_tag = "all" if MAX_IMAGES <= 0 else f"n{MAX_IMAGES}"
clip_tag = CLIP_SIM_MODE + ("_nocrop" if CLIP_NO_CROP else "")
OUTDIR = f"fsc147_{SPLIT}_{FSOC.lower()}_{SSES_MODE}_{clip_tag}_{SAMPLE_MODE}_{n_tag}"

flags = [
    f"--data '{DRIVE_DATA_DIR}'",
    f"--split {SPLIT}",
    f"--fsoc {FSOC}",
    f"--outdir '{OUTDIR}'",
    f"--max_images {MAX_IMAGES}",
    f"--sample_mode {SAMPLE_MODE}",
    f"--sample_seed {SAMPLE_SEED}",
    f"--stage_samples {STAGE_SAMPLES}",
    f"--clip_sim_mode {CLIP_SIM_MODE}",
    "--single_instance_mode paper",
]
if CLIP_NO_CROP:
    flags.append("--clip_no_crop")
if SAVE_FINAL_VIS:
    flags.append("--visualize")
if SSES_MODE == "paper":
    flags.extend(["--box_threshold 0.15", "--text_threshold 0.15", "--sses_start_percentile 90"])
else:
    flags.extend(["--github_sses_inversion", "--box_threshold 0.20", "--text_threshold 0.20", "--sses_start_percentile 80"])

cmd = "python countzes_fsc147.py " + " ".join(flags)
print("Running:", cmd)
!{cmd}

In [ ]:
# [CELL-8] Kiểm tra bằng chứng, log tài nguyên, thống kê theo class & ảnh phân tích nhiều bước
import glob, json
import pandas as pd
from IPython.display import Image, display

run_dir = os.path.join(DRIVE_OUTPUTS_DIR, OUTDIR)
summary_path = os.path.join(run_dir, "summary.json")
res_log_path = os.path.join(run_dir, "logs", "resource_usage.log")
csv_path = os.path.join(run_dir, "predictions.csv")

if os.path.exists(summary_path):
    summary = json.load(open(summary_path, encoding="utf-8"))
    print("=== SUMMARY ===")
    brief = {k: v for k, v in summary.items() if k not in ("per_class_metrics", "failed_files", "resource_eval")}
    print(json.dumps(brief, indent=2, ensure_ascii=False))
    if summary.get("per_class_metrics"):
        print("\n=== PER-CLASS MAE / RMSE (xếp từ sai số cao -> thấp) ===")
        df_cls = pd.DataFrame.from_dict(summary["per_class_metrics"], orient="index")
        display(df_cls)

if os.path.exists(res_log_path):
    print("=== RESOURCE LOG ===")
    print(open(res_log_path, encoding="utf-8").read())

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    cols = [c for c in ["filename", "class_name", "gt_count", "pred_count", "abs_error", "gdino_boxes_n", "count_stage2", "pseudo_gt_kde", "count_stage3", "single2_n", "duration_sec", "status"] if c in df.columns]
    print(f"=== TOP 5 ẢNH SAI SỐ LỚN NHẤT ({len(df)} ảnh) ===")
    display(df.sort_values("abs_error", ascending=False)[cols].head(5))
    print("=== TOP 5 ẢNH DỰ ĐOÁN CHÍNH XÁC NHẤT ===")
    display(df.sort_values("abs_error", ascending=True)[cols].head(5))

stage_imgs = sorted(glob.glob(f"{run_dir}/stage_samples/*.png"))[:10]
print(f"Hiển thị {len(stage_imgs)}/{len(glob.glob(f'{run_dir}/stage_samples/*.png'))} ảnh phân tích đa bước (2x4) tại {run_dir}/stage_samples/:")
for p in stage_imgs:
    print(os.path.basename(p))
    display(Image(filename=p))